In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
import warnings, os, time
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [23]:
sales      = pd.read_csv(f"../data/processed/sales.csv", parse_dates=["Date"])
promotions = pd.read_csv(f"../data/processed/promotions.csv",
                             parse_dates=["start_date", "end_date"])
web        = pd.read_csv(f"../data/processed/web_traffic.csv", parse_dates=["date"])
sample_sub = pd.read_csv(f"../data/processed/sample_submission.csv", parse_dates=["Date"])

print(f"Train rows: {len(sales):,} ({sales['Date'].min().date()} => {sales['Date'].max().date()})")
print(f"Test rows: {len(sample_sub):,} ({sample_sub['Date'].min().date()} => {sample_sub['Date'].max().date()})")
print(f"Promotions: {len(promotions):,}")
print(f"Web traffic: {len(web):,}")
sales.head()

Train rows: 3,833 (2012-07-04 => 2022-12-31)
Test rows: 548 (2023-01-01 => 2024-07-01)
Promotions: 50
Web traffic: 3,652


,Date,Revenue,COGS,is_holiday,log_Revenue,log_COGS
0,2012-07-04,5123547.94,3982991.19,0,15.449358,15.197544
1,2012-07-05,2751773.45,2150580.23,0,14.827757,14.581249
2,2012-07-06,3054029.42,2517632.84,0,14.931973,14.738830
3,2012-07-07,2667930.94,2108246.62,0,14.796814,14.561368
4,2012-07-08,2360851.90,1808622.79,0,14.674534,14.408077


## Feature Engineering

In [24]:
def build_features(df: pd.DataFrame,
                   promotions: pd.DataFrame,
                   web: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").reset_index(drop=True).copy()

    # Calendar
    df["dow"]       = df["Date"].dt.dayofweek
    df["month"]     = df["Date"].dt.month
    df["dom"]       = df["Date"].dt.day
    df["woy"]       = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"]   = df["Date"].dt.quarter
    df["is_weekend"]= df["dow"].isin([5, 6]).astype(int)
    df["year"]      = df["Date"].dt.year
    df["year_norm"] = (df["year"] - 2012) / 10.0 # xu huong tang truong

    # Tet
    tet_dates = pd.to_datetime([
        "2012-01-23","2013-02-10","2014-01-31","2015-02-19","2016-02-08",
        "2017-01-28","2018-02-16","2019-02-05","2020-01-25","2021-02-12",
        "2022-02-01","2023-01-22","2024-02-10","2025-01-29"
    ])
    df["days_to_tet"] = df["Date"].apply(
        lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=60)
    df["pre_tet_7d"]  = df["Date"].apply(
        lambda x: int(any(0  < (t - x).days <= 7  for t in tet_dates)))
    df["pre_tet_14d"] = df["Date"].apply(
        lambda x: int(any(7  < (t - x).days <= 14 for t in tet_dates)))
    df["pre_tet_30d"] = df["Date"].apply(
        lambda x: int(any(14 < (t - x).days <= 30 for t in tet_dates)))
    df["on_tet"]      = df["Date"].apply(
        lambda x: int(any(abs((x - t).days) <= 2  for t in tet_dates)))
    df["post_tet"]    = df["Date"].apply(
        lambda x: int(any(0  < (x - t).days <= 7  for t in tet_dates)))

    # Holidays
    fixed_holidays = []
    for y in range(2012, 2026):
        fixed_holidays += [f"{y}-01-01", f"{y}-04-30", f"{y}-05-01", f"{y}-09-02"]
    gioto = ["2012-03-31","2013-04-19","2014-04-09","2015-04-28","2016-04-16",
             "2017-04-06","2018-04-25","2019-04-14","2020-04-02","2021-04-21",
             "2022-04-10","2023-04-29","2024-04-18","2025-04-07"]
    all_holidays = pd.to_datetime(fixed_holidays + gioto)
    df["is_holiday"] = df["Date"].apply(
        lambda x: int(min(abs((x - h).days) for h in all_holidays) <= 3))

    # Promotion
    df["promo_count"] = 0
    for _, row in promotions.iterrows():
        mask = (df["Date"] >= row["start_date"]) & (df["Date"] <= row["end_date"])
        df.loc[mask, "promo_count"] += 1
    df["promo_active"]    = (df["promo_count"] > 0).astype(int)
    df["promo_intensity"] = df["promo_count"].clip(upper=5)
    df["post_promo"]      = ((df["promo_active"].shift(1).fillna(0) == 1) &
                              (df["promo_active"] == 0)).astype(int)

    # Web traffic
    daily_web = web.groupby("date")["sessions"].sum().reset_index()
    df = df.merge(daily_web, left_on="Date", right_on="date", how="left").drop(columns=["date"])
    df["sessions"] = df["sessions"].fillna(df["sessions"].median())

    roll30_sess = df["sessions"].shift(1).rolling(30).mean()
    df["sessions_lag7"]       = df["sessions"].shift(7)
    df["sessions_lag14"]      = df["sessions"].shift(14)
    df["sessions_roll7_mean"] = df["sessions"].shift(1).rolling(7).mean()
    df["sessions_vs_avg"]     = (df["sessions"] / roll30_sess).fillna(1.0)
    df["sessions_spike"]      = (df["sessions_vs_avg"] > 1.5).astype(int)

    # Revenue lags & rolling
    rev = df["Revenue"]
    for lag in [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 365]:
        df[f"rev_lag_{lag}"] = rev.shift(lag)

    # EWM (Exponentially Weighted Mean) - avg uu tien ngay gan hon
    for span in [7, 14, 30]:
        df[f"rev_ewm{span}"] = rev.shift(1).ewm(span=span, adjust=False).mean()

    for w in [7, 14, 30, 90]:
        df[f"rev_roll{w}_mean"] = rev.shift(1).rolling(w).mean()
        df[f"rev_roll{w}_std"]  = rev.shift(1).rolling(w).std()
        df[f"rev_roll{w}_max"]  = rev.shift(1).rolling(w).max()
        df[f"rev_roll{w}_min"]  = rev.shift(1).rolling(w).min()

    # Year-over-Year - So sanh cung ki nam truoc
    df["rev_yoy_lag365"] = rev.shift(365)
    df["rev_yoy_ratio"]  = (rev.shift(1) / rev.shift(366).replace(0, np.nan)
                            ).fillna(1.0).clip(0.1, 10)

    # COGS lags 
    cogs = df["COGS"]
    for lag in [1, 7, 30, 365]:
        df[f"cogs_lag_{lag}"] = cogs.shift(lag)
    df["cogs_roll7_mean"]  = cogs.shift(1).rolling(7).mean()
    df["cogs_roll30_mean"] = cogs.shift(1).rolling(30).mean()

    # Revenue/COGS ratio (gross margin proxy) - Nhap: 100, Ban 140 -> Tile 1.4
    # The hien hieu qua kinh doanh
    df["rev_cogs_ratio_lag1"] = (
        rev.shift(1) / cogs.shift(1).replace(0, np.nan)
    ).fillna(1.4).clip(0.5, 5)

    return df

print("build_features")

build_features


In [25]:
df_feat = build_features(sales.copy(), promotions, web)

EXCLUDE = {"Date", "Revenue", "COGS", "log_Revenue", "log_COGS",
           "sessions", "promo_count", "promo_active"}

feature_cols = [c for c in df_feat.columns if c not in EXCLUDE]

print(f"Shape:{df_feat.shape}")
print(f"Total features: {len(feature_cols)}")
print(f"\nFeatures list:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

Shape:(3833, 69)
Total features: 61

Features list:
   1. is_holiday
   2. dow
   3. month
   4. dom
   5. woy
   6. quarter
   7. is_weekend
   8. year
   9. year_norm
  10. days_to_tet
  11. pre_tet_7d
  12. pre_tet_14d
  13. pre_tet_30d
  14. on_tet
  15. post_tet
  16. promo_intensity
  17. post_promo
  18. sessions_lag7
  19. sessions_lag14
  20. sessions_roll7_mean
  21. sessions_vs_avg
  22. sessions_spike
  23. rev_lag_1
  24. rev_lag_2
  25. rev_lag_3
  26. rev_lag_7
  27. rev_lag_14
  28. rev_lag_21
  29. rev_lag_28
  30. rev_lag_30
  31. rev_lag_60
  32. rev_lag_90
  33. rev_lag_365
  34. rev_ewm7
  35. rev_ewm14
  36. rev_ewm30
  37. rev_roll7_mean
  38. rev_roll7_std
  39. rev_roll7_max
  40. rev_roll7_min
  41. rev_roll14_mean
  42. rev_roll14_std
  43. rev_roll14_max
  44. rev_roll14_min
  45. rev_roll30_mean
  46. rev_roll30_std
  47. rev_roll30_max
  48. rev_roll30_min
  49. rev_roll90_mean
  50. rev_roll90_std
  51. rev_roll90_max
  52. rev_roll90_min
  53. rev_yoy_la

In [26]:
def get_lgb_params(target="Revenue"):
    return dict(
        n_estimators      = 3000,
        learning_rate     = 0.02,
        num_leaves        = 63,
        max_depth         = -1,
        min_child_samples = 20,
        feature_fraction  = 0.8,
        bagging_fraction  = 0.8,
        bagging_freq      = 5,
        reg_alpha         = 0.1,
        reg_lambda        = 0.2,
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1,
    )

def get_xgb_params():
    return dict(
        n_estimators     = 2000,
        learning_rate    = 0.03,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        random_state     = SEED,
        verbosity        = 0,
    )

def walk_forward_cv(df_train, feature_cols, target="Revenue", n_splits=5, min_train_days=365):
    df_train = df_train.dropna(subset=feature_cols+[target]).reset_index(drop=True)
    n = len(df_train)
    fold_size = (n - min_train_days) // n_splits
    oof_lgb  = np.zeros(n)
    oof_xgb  = np.zeros(n)
    oof_mask = np.zeros(n, dtype=bool)
    best_iters = []
    for i in range(n_splits):
        cut     = min_train_days + i * fold_size
        val_end = cut + fold_size if i < n_splits-1 else n
        X_tr, y_tr   = df_train.iloc[:cut][feature_cols], df_train.iloc[:cut][target]
        X_val, y_val = df_train.iloc[cut:val_end][feature_cols], df_train.iloc[cut:val_end][target]
        # LightGBM with early stopping
        m_lgb = lgb.LGBMRegressor(**get_lgb_params())
        m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        best_iters.append(m_lgb.best_iteration_)
        # XGBoost
        m_xgb = xgb.XGBRegressor(**get_xgb_params())
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        p_lgb = m_lgb.predict(X_val)
        p_xgb = m_xgb.predict(X_val)
        mae_lgb = mean_absolute_error(y_val, p_lgb)
        mae_xgb = mean_absolute_error(y_val, p_xgb)
        print(f"  Fold {i+1} | LGB MAE={mae_lgb:,.0f} | XGB MAE={mae_xgb:,.0f} | best_iter={best_iters[-1]}")
        oof_lgb[cut:val_end] = p_lgb
        oof_xgb[cut:val_end] = p_xgb
        oof_mask[cut:val_end] = True
    avg_iter = int(np.mean(best_iters))
    y_oof = df_train[target].values[oof_mask]
    print(f"OOF LGB MAE={mean_absolute_error(y_oof, oof_lgb[oof_mask]):,.0f}")
    print(f"OOF XGB MAE={mean_absolute_error(y_oof, oof_xgb[oof_mask]):,.0f}")
    print(f"Avg best_iteration: {avg_iter}")
    return oof_lgb, oof_xgb, oof_mask, avg_iter

print("walk_forward_cv")

walk_forward_cv


In [27]:
print("Walk-forward CV — REVENUE")
oof_lgb_r, oof_xgb_r, mask_r, best_iter_r = walk_forward_cv(df_feat, feature_cols, "Revenue")
print("Walk-forward CV — COGS")
oof_lgb_c, oof_xgb_c, mask_c, best_iter_c = walk_forward_cv(df_feat, feature_cols, "COGS")


Walk-forward CV — REVENUE
  Fold 1 | LGB MAE=837,866 | XGB MAE=823,321 | best_iter=240
  Fold 2 | LGB MAE=987,537 | XGB MAE=1,023,221 | best_iter=200
  Fold 3 | LGB MAE=845,923 | XGB MAE=983,156 | best_iter=205
  Fold 4 | LGB MAE=518,272 | XGB MAE=546,398 | best_iter=257
  Fold 5 | LGB MAE=546,263 | XGB MAE=541,497 | best_iter=374
OOF LGB MAE=746,978
OOF XGB MAE=783,285
Avg best_iteration: 255
Walk-forward CV — COGS
  Fold 1 | LGB MAE=715,134 | XGB MAE=745,767 | best_iter=234
  Fold 2 | LGB MAE=855,597 | XGB MAE=894,611 | best_iter=273
  Fold 3 | LGB MAE=800,346 | XGB MAE=892,515 | best_iter=200
  Fold 4 | LGB MAE=458,220 | XGB MAE=479,234 | best_iter=233
  Fold 5 | LGB MAE=479,135 | XGB MAE=484,374 | best_iter=320
OOF LGB MAE=661,510
OOF XGB MAE=699,093
Avg best_iteration: 252


In [28]:
def find_weights_oof(oof_lgb, oof_xgb, y_true, mask):
    p_l, p_x, y = oof_lgb[mask], oof_xgb[mask], y_true[mask]
    def obj(w):
        w_n = np.exp(w) / np.exp(w).sum()
        return mean_absolute_error(y, w_n[0]*p_l + w_n[1]*p_x)  # Dùng MAE vì Kaggle dùng MAE
    res = minimize(obj, [0.0, 0.0], method="Nelder-Mead", options={"maxiter":5000})
    w = np.exp(res.x) / np.exp(res.x).sum()
    print(f"  Weights: LGB={w[0]:.4f} | XGB={w[1]:.4f} | OOF MAE={res.fun:,.0f}")
    return float(w[0]), float(w[1])
df_clean = df_feat.dropna(subset=feature_cols+["Revenue","COGS"]).reset_index(drop=True)
y_all = df_clean[["Revenue","COGS"]].values
print("Revenue weights:")
w_lgb_r, w_xgb_r = find_weights_oof(oof_lgb_r, oof_xgb_r, df_clean["Revenue"].values, mask_r)
print("COGS weights:")
w_lgb_c, w_xgb_c = find_weights_oof(oof_lgb_c, oof_xgb_c, df_clean["COGS"].values, mask_c)


Revenue weights:
  Weights: LGB=0.9934 | XGB=0.0066 | OOF MAE=746,973
COGS weights:
  Weights: LGB=1.0000 | XGB=0.0000 | OOF MAE=661,510


In [29]:
def train_final(df_full, feature_cols, target, best_iter):
    df_full = df_full.dropna(subset=feature_cols+[target]).copy()
    X, y = df_full[feature_cols], df_full[target]
    print(f"  Training {target} on {len(X):,} samples | n_estimators(LGB)={best_iter}")
    lgb_p = get_lgb_params()
    lgb_p["n_estimators"] = best_iter
    m_lgb = lgb.LGBMRegressor(**lgb_p)
    m_lgb.fit(X, y, callbacks=[lgb.log_evaluation(-1)])
    m_xgb = xgb.XGBRegressor(**get_xgb_params())
    m_xgb.fit(X, y)
    return m_lgb, m_xgb
lgb_rev, xgb_rev   = train_final(df_feat, feature_cols, "Revenue", best_iter_r)
lgb_cogs, xgb_cogs = train_final(df_feat, feature_cols, "COGS", best_iter_c)

  Training Revenue on 3,468 samples | n_estimators(LGB)=255
  Training COGS on 3,468 samples | n_estimators(LGB)=252


In [30]:
'''ver 4:
Test có 548 ngày, nhưng lag features (rev_lag_1, rev_lag_7...)
đều = NaN vì không có Revenue/COGS thật ở test period.
V4 fill NaN bằng median → model mất thông tin quan trọng nhất.

def build_test_features(df_train_full, sample_sub, promotions, web, feature_cols):
    test_stub = sample_sub[["Date"]].copy()
    test_stub["Revenue"] = np.nan
    test_stub["COGS"]    = np.nan

    df_combined = pd.concat(
        [df_train_full[["Date", "Revenue", "COGS"]], test_stub],
        ignore_index=True
    )
    df_combined = build_features(df_combined, promotions, web)

    df_test = df_combined[df_combined["Date"] >= sample_sub["Date"].min()].copy()
    X_test  = df_test[feature_cols]
    return X_test, df_test

X_test, df_test = build_test_features(sales, sample_sub, promotions, web, feature_cols)

nan_count = X_test.isna().sum().sum()
print(f"Test shape: {X_test.shape}")
print(f"NaN count: {nan_count}")

if nan_count > 0:
    print("\nNaN per column:")
    nan_cols = X_test.isna().sum()
    print(nan_cols[nan_cols > 0])

    # Fill remaining NaN với median của train
    train_medians = df_clean[feature_cols].median()
    X_test = X_test.fillna(train_medians)
    print(f"\nAfter fillna: NaN = {X_test.isna().sum().sum()}")

print("\nTest features ready")
'''

'ver 4:\nTest có 548 ngày, nhưng lag features (rev_lag_1, rev_lag_7...)\nđều = NaN vì không có Revenue/COGS thật ở test period.\nV4 fill NaN bằng median → model mất thông tin quan trọng nhất.\n\ndef build_test_features(df_train_full, sample_sub, promotions, web, feature_cols):\n    test_stub = sample_sub[["Date"]].copy()\n    test_stub["Revenue"] = np.nan\n    test_stub["COGS"]    = np.nan\n\n    df_combined = pd.concat(\n        [df_train_full[["Date", "Revenue", "COGS"]], test_stub],\n        ignore_index=True\n    )\n    df_combined = build_features(df_combined, promotions, web)\n\n    df_test = df_combined[df_combined["Date"] >= sample_sub["Date"].min()].copy()\n    X_test  = df_test[feature_cols]\n    return X_test, df_test\n\nX_test, df_test = build_test_features(sales, sample_sub, promotions, web, feature_cols)\n\nnan_count = X_test.isna().sum().sum()\nprint(f"Test shape: {X_test.shape}")\nprint(f"NaN count: {nan_count}")\n\nif nan_count > 0:\n    print("\nNaN per column:")\n   

In [31]:
def recursive_forecast(lgb_r, xgb_r, lgb_c, xgb_c,
                       df_train, sample_sub, promotions, web, feature_cols,
                       w_lr, w_xr, w_lc, w_xc, train_medians):
    test_stub = sample_sub[["Date"]].copy()
    test_stub["Revenue"] = np.nan
    test_stub["COGS"] = np.nan
    df_all = pd.concat([df_train[["Date","Revenue","COGS"]], test_stub], ignore_index=True)
    df_all = build_features(df_all, promotions, web)
    print(f"Combined shape: {df_all.shape}")

    all_rev = df_all["Revenue"].values.astype(float).copy()
    all_cogs = df_all["COGS"].values.astype(float).copy()
    train_n = len(df_train)

    col_idx = {c: i for i, c in enumerate(feature_cols)}
    
    test_X = df_all.iloc[train_n:][feature_cols].values.astype(float).copy()
    
    rev_lags  = [(f"rev_lag_{l}", l) for l in [1,2,3,7,14,21,28,30,60,90,365]]
    cogs_lags = [(f"cogs_lag_{l}", l) for l in [1,7,30,365]]
    t0 = time.time()
    for i in range(len(sample_sub)):
        idx = train_n + i  

        for col_name, lag in rev_lags:
            if col_name in col_idx and idx >= lag:
                val = all_rev[idx - lag]
                if not np.isnan(val):
                    test_X[i, col_idx[col_name]] = val
        
        for w in [7, 14, 30, 90]:
            start = max(0, idx - w)
            vals = all_rev[start:idx]
            valid = vals[~np.isnan(vals)]
            if len(valid) > 0:
                if f"rev_roll{w}_mean" in col_idx:
                    test_X[i, col_idx[f"rev_roll{w}_mean"]] = np.mean(valid)
                if f"rev_roll{w}_std" in col_idx:
                    test_X[i, col_idx[f"rev_roll{w}_std"]] = np.std(valid, ddof=1) if len(valid)>1 else 0
                if f"rev_roll{w}_max" in col_idx:
                    test_X[i, col_idx[f"rev_roll{w}_max"]] = np.max(valid)
                if f"rev_roll{w}_min" in col_idx:
                    test_X[i, col_idx[f"rev_roll{w}_min"]] = np.min(valid)
        
        for span in [7, 14, 30]:
            cn = f"rev_ewm{span}"
            if cn in col_idx:
                vals = all_rev[max(0, idx-span*3):idx]
                valid = vals[~np.isnan(vals)]
                if len(valid) > 0:
                    alpha = 2.0 / (span + 1)
                    ewm = valid[0]
                    for v in valid[1:]:
                        ewm = alpha * v + (1 - alpha) * ewm
                    test_X[i, col_idx[cn]] = ewm
        
        if "rev_yoy_lag365" in col_idx and idx >= 365:
            v = all_rev[idx-365]
            if not np.isnan(v):
                test_X[i, col_idx["rev_yoy_lag365"]] = v
        if "rev_yoy_ratio" in col_idx and idx >= 366:
            prev, prev_yoy = all_rev[idx-1], all_rev[idx-366]
            if not np.isnan(prev) and not np.isnan(prev_yoy) and prev_yoy != 0:
                test_X[i, col_idx["rev_yoy_ratio"]] = np.clip(prev/prev_yoy, 0.1, 10)
        
        for col_name, lag in cogs_lags:
            if col_name in col_idx and idx >= lag:
                val = all_cogs[idx - lag]
                if not np.isnan(val):
                    test_X[i, col_idx[col_name]] = val
        for w, cn in [(7,"cogs_roll7_mean"),(30,"cogs_roll30_mean")]:
            if cn in col_idx:
                vals = all_cogs[max(0,idx-w):idx]
                valid = vals[~np.isnan(vals)]
                if len(valid) > 0:
                    test_X[i, col_idx[cn]] = np.mean(valid)
        
        if "rev_cogs_ratio_lag1" in col_idx and idx >= 1:
            pr, pc = all_rev[idx-1], all_cogs[idx-1]
            if not np.isnan(pr) and not np.isnan(pc) and pc != 0:
                test_X[i, col_idx["rev_cogs_ratio_lag1"]] = np.clip(pr/pc, 0.5, 5)
        
        for j in range(len(feature_cols)):
            if np.isnan(test_X[i, j]):
                test_X[i, j] = train_medians.iloc[j]
        
        X_row = pd.DataFrame([test_X[i]], columns=feature_cols)
        r = w_lr * lgb_r.predict(X_row)[0] + w_xr * xgb_r.predict(X_row)[0]
        c = w_lc * lgb_c.predict(X_row)[0] + w_xc * xgb_c.predict(X_row)[0]
        all_rev[idx]  = max(r, 0)
        all_cogs[idx] = max(c, 0)
        if (i+1) % 100 == 0:
            print(f"Day {i+1}/548 | Rev={all_rev[idx]:,.0f} | COGS={all_cogs[idx]:,.0f} | {time.time()-t0:.1f}s")
    print(f"Recursive forecast done in {time.time()-t0:.1f}s")
    return all_rev[train_n:], all_cogs[train_n:]

train_medians = df_clean[feature_cols].median()
print("\nRECURSIVE FORECASTING:")
pred_rev, pred_cog = recursive_forecast(
    lgb_rev, xgb_rev, lgb_cogs, xgb_cogs,
    sales, sample_sub, promotions, web, feature_cols,
    w_lgb_r, w_xgb_r, w_lgb_c, w_xgb_c, train_medians
)
pred_rev = np.round(np.clip(pred_rev, 0, None), 2)
pred_cog = np.round(np.clip(pred_cog, 0, None), 2)
print(f"\nRevenue: min={pred_rev.min():,.0f} | mean={pred_rev.mean():,.0f} | max={pred_rev.max():,.0f}")
print(f"COGS: min={pred_cog.min():,.0f} | mean={pred_cog.mean():,.0f} | max={pred_cog.max():,.0f}")



RECURSIVE FORECASTING:
Combined shape: (4381, 67)
Day 100/548 | Rev=3,562,767 | COGS=3,287,553 | 1.2s
Day 200/548 | Rev=4,189,709 | COGS=3,374,680 | 2.5s
Day 300/548 | Rev=3,778,842 | COGS=3,235,674 | 3.8s
Day 400/548 | Rev=2,365,181 | COGS=1,991,414 | 4.9s
Day 500/548 | Rev=5,151,509 | COGS=4,135,719 | 6.2s
Recursive forecast done in 6.9s

Revenue: min=1,031,470 | mean=4,036,982 | max=10,174,740
COGS: min=777,224 | mean=3,323,271 | max=9,783,149


In [32]:
'''
def ensemble_predict(lgb_model, xgb_model, X, w_lgb, w_xgb):
    p_lgb = lgb_model.predict(X)
    p_xgb = xgb_model.predict(X)
    return w_lgb * p_lgb + w_xgb * p_xgb

pred_rev = ensemble_predict(lgb_rev, xgb_rev,   X_test, w_lgb_r, w_xgb_r)
pred_cog = ensemble_predict(lgb_cogs, xgb_cogs, X_test, w_lgb_c, w_xgb_c)

pred_rev = np.round(np.clip(pred_rev, 0, None), 2)
pred_cog = np.round(np.clip(pred_cog, 0, None), 2)

print("Revenue predictions:")
print(f"min={pred_rev.min():,.0f} | mean={pred_rev.mean():,.0f} | max={pred_rev.max():,.0f}")
print("\nCOGS predictions:")
print(f"min={pred_cog.min():,.0f} | mean={pred_cog.mean():,.0f} | max={pred_cog.max():,.0f}")
'''

'\ndef ensemble_predict(lgb_model, xgb_model, X, w_lgb, w_xgb):\n    p_lgb = lgb_model.predict(X)\n    p_xgb = xgb_model.predict(X)\n    return w_lgb * p_lgb + w_xgb * p_xgb\n\npred_rev = ensemble_predict(lgb_rev, xgb_rev,   X_test, w_lgb_r, w_xgb_r)\npred_cog = ensemble_predict(lgb_cogs, xgb_cogs, X_test, w_lgb_c, w_xgb_c)\n\npred_rev = np.round(np.clip(pred_rev, 0, None), 2)\npred_cog = np.round(np.clip(pred_cog, 0, None), 2)\n\nprint("Revenue predictions:")\nprint(f"min={pred_rev.min():,.0f} | mean={pred_rev.mean():,.0f} | max={pred_rev.max():,.0f}")\nprint("\nCOGS predictions:")\nprint(f"min={pred_cog.min():,.0f} | mean={pred_cog.mean():,.0f} | max={pred_cog.max():,.0f}")\n'

In [33]:
'''import os

sub = sample_sub.copy()
sub["Revenue"] = pred_rev
sub["COGS"] = pred_cog

assert len(sub) == 548, f"ERROR: expected 548 rows, got {len(sub)}"

out_path = f"../submissions/submission_v4.csv"
sub.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(sub)}")
sub.head(10)'''

'import os\n\nsub = sample_sub.copy()\nsub["Revenue"] = pred_rev\nsub["COGS"] = pred_cog\n\nassert len(sub) == 548, f"ERROR: expected 548 rows, got {len(sub)}"\n\nout_path = f"../submissions/submission_v4.csv"\nsub.to_csv(out_path, index=False)\n\nprint(f"Saved: {out_path}")\nprint(f"Rows: {len(sub)}")\nsub.head(10)'

In [34]:
'''import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Revenue
ax = axes[0]
ax.plot(sales["Date"], sales["Revenue"], alpha=0.6, label="Train Revenue", color="steelblue")
ax.plot(sub["Date"], sub["Revenue"], color="tomato", linewidth=2, label="Predicted Revenue (v4)")
ax.set_title("Revenue: Train vs Prediction", fontsize=13)
ax.legend()
ax.set_ylabel("Revenue (VND)")
ax.grid(alpha=0.3)

# COGS
ax = axes[1]
ax.plot(sales["Date"], sales["COGS"], alpha=0.6, label="Train COGS", color="darkorange")
ax.plot(sub["Date"], sub["COGS"], color="purple", linewidth=2, label="Predicted COGS (v4)")
ax.set_title("COGS: Train vs Prediction", fontsize=13)
ax.legend()
ax.set_ylabel("COGS (VND)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"../report/figures/prediction_plot_v4.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved")'''

'import matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nfig, axes = plt.subplots(2, 1, figsize=(14, 8))\n\n# Revenue\nax = axes[0]\nax.plot(sales["Date"], sales["Revenue"], alpha=0.6, label="Train Revenue", color="steelblue")\nax.plot(sub["Date"], sub["Revenue"], color="tomato", linewidth=2, label="Predicted Revenue (v4)")\nax.set_title("Revenue: Train vs Prediction", fontsize=13)\nax.legend()\nax.set_ylabel("Revenue (VND)")\nax.grid(alpha=0.3)\n\n# COGS\nax = axes[1]\nax.plot(sales["Date"], sales["COGS"], alpha=0.6, label="Train COGS", color="darkorange")\nax.plot(sub["Date"], sub["COGS"], color="purple", linewidth=2, label="Predicted COGS (v4)")\nax.set_title("COGS: Train vs Prediction", fontsize=13)\nax.legend()\nax.set_ylabel("COGS (VND)")\nax.grid(alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(f"../report/figures/prediction_plot_v4.png", dpi=120, bbox_inches="tight")\nplt.show()\nprint("Plot saved")'

In [35]:
'''fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, title in zip(axes,
                             [lgb_rev, lgb_cogs],
                             ["Revenue", "COGS"]):
    importance = pd.Series(model.feature_importances_, index=feature_cols)
    top20 = importance.nlargest(20)
    top20[::-1].plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"LGB Feature Importance — {title}", fontsize=12)
    ax.set_xlabel("Importance")
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(f"../report/figures/feature_importance_v4.png", dpi=120, bbox_inches="tight")
plt.show()
print("Feature importance saved")'''

'fig, axes = plt.subplots(1, 2, figsize=(16, 8))\n\nfor ax, model, title in zip(axes,\n                             [lgb_rev, lgb_cogs],\n                             ["Revenue", "COGS"]):\n    importance = pd.Series(model.feature_importances_, index=feature_cols)\n    top20 = importance.nlargest(20)\n    top20[::-1].plot(kind="barh", ax=ax, color="steelblue")\n    ax.set_title(f"LGB Feature Importance — {title}", fontsize=12)\n    ax.set_xlabel("Importance")\n    ax.grid(axis="x", alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(f"../report/figures/feature_importance_v4.png", dpi=120, bbox_inches="tight")\nplt.show()\nprint("Feature importance saved")'

In [36]:
sub = sample_sub.copy()
sub["Revenue"] = pred_rev
sub["COGS"] = pred_cog
assert len(sub) == 548, f"ERROR: {len(sub)} rows"
out_path = f"../submissions/submission_v5.csv"
sub.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Rows: {len(sub)}")
print(sub.head(10))

Saved: ../submissions/submission_v5.csv
Rows: 548
        Date     Revenue        COGS
0 2023-01-01  2009048.66  1650650.35
1 2023-01-02  1983104.15  1545448.43
2 2023-01-03  1432611.92  1240016.20
3 2023-01-04  1031470.49   777224.44
4 2023-01-05  1248766.58  1029135.50
5 2023-01-06  1384765.56  1096534.37
6 2023-01-07  1522140.95  1234458.97
7 2023-01-08  1798605.71  1447227.20
8 2023-01-09  1961846.83  1646804.93
9 2023-01-10  2044390.62  1751843.70
